In [1]:
import liana as li
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import anndata as ad

In [4]:
import os
os.chdir('workspace/PDAC_Work_Dir/PDAC_Final/Downstream')

In [5]:
adata = sc.read_h5ad('radiotherapy.h5ad')

In [6]:
adata

In [ ]:
li.mt.show_methods()

In [ ]:
from liana.method import singlecellsignalr, connectome, cellphonedb, natmi, logfc, cellchat, geometric_mean

In [ ]:
adata.X = adata.layers['log_norm'].copy()

# RT

In [ ]:
radio = adata[adata.obs.Radiotherapy_Status == 'Radiotherapy']

In [ ]:
li.mt.rank_aggregate(radio, 
                     groupby='Level_4', use_raw=False, layer='log_norm',
                     resource_name='consensus',
                     expr_prop=0.1,
                     verbose=True)

In [ ]:
radio

In [ ]:
df = radio.uns['liana_res']

In [ ]:
df.head()

In [ ]:
df_subset = df[~(df.source.isin(['Alpha Cell', 'Beta Cell', 'Other Endocrine']))]

In [ ]:
df_subset = df_subset[~(df_subset.target.isin(['Alpha Cell', 'Beta Cell', 'Other Endocrine']))]

In [ ]:
df.to_csv('cellcell_comm_radio_all_resources.csv')

In [ ]:
df[df.source.str.contains('Malignant') & (df.target.str.contains('Endo'))].head(100)

# NO-RT

In [ ]:
non_radio = adata[adata.obs.Radiotherapy_Status != 'Radiotherapy']

In [ ]:
non_radio.obs.Radiotherapy_Status.value_counts()

In [ ]:
li.mt.rank_aggregate(non_radio, 
                     groupby='Level_4', use_raw=False, layer='log_norm',
                     resource_name='consensus',
                     expr_prop=0.1,
                     verbose=True)

In [ ]:
df_non_radio = non_radio.uns['liana_res']

In [ ]:
df_non_radio.to_csv('cellcell_comm_non_radio_all_resources.csv')

In [ ]:
df_non_radio.head()

# Reload

In [7]:
df_non_radio = pd.read_csv('cellcell_comm_non_radio_all_resources.csv')
df_radio = pd.read_csv('cellcell_comm_radio_all_resources.csv')

In [8]:
pwd

In [9]:
radio = adata[adata.obs.Radiotherapy_Status == 'Radiotherapy']
non_radio = adata[adata.obs.Radiotherapy_Status != 'Radiotherapy']

In [10]:
radio.uns['liana_res'] = df_radio.copy()
non_radio.uns['liana_res'] = df_non_radio.copy()

In [11]:
keep_cols = ["source","target","ligand_complex","receptor_complex",
             "magnitude_rank","specificity_rank","lr_probs","lr_logfc",
             "cellphone_pvals","cellchat_pvals"]

r = df_radio[keep_cols].rename(columns=lambda c: c if c in ["source","target","ligand_complex","receptor_complex"] else f"{c}_rt")
n = df_non_radio[keep_cols].rename(columns=lambda c: c if c in ["source","target","ligand_complex","receptor_complex"] else f"{c}_nor")

cmp = (r.merge(n, on=["source","target","ligand_complex","receptor_complex"], how="inner")
         .assign(
           d_mag = lambda d: d["magnitude_rank_nor"] - d["magnitude_rank_rt"], # +ve => stronger in RT
           d_spec= lambda d: d["specificity_rank_nor"] - d["specificity_rank_rt"],
           d_prob= lambda d: d["lr_probs_rt"] - d["lr_probs_nor"],
           d_logfc=lambda d: d["lr_logfc_rt"] - d["lr_logfc_nor"],
           sig_rt = lambda d: (d["cellphone_pvals_rt"]<0.05) | (d["cellchat_pvals_rt"]<0.05),
           sig_nor= lambda d: (d["cellphone_pvals_nor"]<0.05) | (d["cellchat_pvals_nor"]<0.05),
         ))
focus = cmp.query("(sig_rt | sig_nor) and (d_mag>0.1 or d_prob>0.05 or d_logfc>0.5)")

In [12]:
pd.set_option('display.max_columns', None)

In [13]:
focus

In [14]:
focus["p_rt"]  = focus[["cellphone_pvals_rt","cellchat_pvals_rt"]].min(axis=1)
focus["p_nor"] = focus[["cellphone_pvals_nor","cellchat_pvals_nor"]].min(axis=1)
for c in ["d_mag","d_prob","d_logfc"]:
    mu, sd = focus[c].mean(), focus[c].std(ddof=0) or 1.0
    focus[c+"_z"] = (focus[c] - mu) / sd
focus["delta_score"] = (1.0*focus["d_mag_z"] + 0.7*focus["d_prob_z"] + 0.5*focus["d_logfc_z"])
meaningful = focus.query("(p_rt < 0.05) and (delta_score > 0)")
top_global = meaningful.sort_values(["delta_score","d_mag","d_prob"], ascending=False).head(100)

In [15]:
top_global.head(10)

In [16]:
adata.obs.Level_3.value_counts()

In [17]:
tumor = adata[adata.obs.Level_2.str.contains('Mal')].obs.Level_4.unique().tolist()
cafs  = adata[adata.obs.Level_2.str.contains('Stromal')].obs.Level_4.unique().tolist()
endo  = adata[adata.obs.Level_2.str.contains('Endoth')].obs.Level_4.unique().tolist()
myeloid = adata[adata.obs.Level_2.str.contains('Myelo')].obs.Level_4.unique().tolist()
t_cells = adata[adata.obs.Level_3.str.contains('T Cell')].obs.Level_4.unique().tolist()

axes = {
  "Tumor/Endo":  lambda d: (d.source.isin(tumor)) & (d.target.isin(endo)),
  "Tumor/CAF→Endo":  lambda d: (d.source.isin(tumor+cafs)) & (d.target.isin(endo)),
  "Tumor/CAF↔Tumor": lambda d: (d.source.isin(tumor+cafs)) & (d.target.isin(tumor+cafs)),
  "Myeloid/Endo→T":  lambda d: (d.source.isin(myeloid+endo)) & (d.target.isin(lymphoid)),
  # "Tumor/Lymp":  lambda d: (d.source.isin(tumor)) & (d.target.isin(lymphoid)),
  "Tumor/CAF":  lambda d: (d.source.isin(tumor)) & (d.target.isin(cafs)),
  "Tumor/T":  lambda d: (d.source.isin(tumor)) & (d.target.isin(t_cells)),
}

In [18]:
def top_by_axis(df, mask, k=30):
    d = df[mask(df)].sort_values(["delta_score","d_mag","d_prob"], ascending=[False,False,False]).copy()
    return d[["source","target","ligand_complex","receptor_complex",
              "delta_score","d_mag","d_prob","d_logfc","p_rt","p_nor",
              "magnitude_rank_rt","magnitude_rank_nor","lr_probs_rt","lr_probs_nor"]].head(k)

top_tumor_endo   = top_by_axis(meaningful, axes["Tumor/Endo"], 1000)
top_tumor_caf_endo   = top_by_axis(meaningful, axes["Tumor/CAF→Endo"], 1000)
top_tumor_tumor  = top_by_axis(meaningful, axes["Tumor/CAF↔Tumor"], 1000)
# top_myeloid_t    = top_by_axis(meaningful, axes["Myeloid/Endo→T"], 1000)
top_tumor_t    = top_by_axis(meaningful, axes["Tumor/T"], 1000)
top_tumor_caf    = top_by_axis(meaningful, axes["Tumor/CAF"], 1000)

In [19]:
top_tumor_endo.shape

In [20]:
top_tumor_endo.head(20)

In [21]:
order = ["delta_score", "d_mag", "d_prob"]
sorted_df = top_tumor_endo.sort_values(order, ascending=[False, False, False])
mask = ~sorted_df.duplicated(
    subset=["source", "target", "ligand_complex", "receptor_complex"],
    keep="first"
)

topN_per_source_endo = (sorted_df.loc[mask]
                   .groupby("source", group_keys=False)
                   .head(5)
                   .sort_values("source"))

In [22]:
topN_per_source_endo

In [23]:
top_tumor_t

In [24]:
order = ["delta_score", "d_mag", "d_prob"]
sorted_df = top_tumor_t.sort_values(order, ascending=[False, False, False])
mask = ~sorted_df.duplicated(
    subset=["source", "target", "ligand_complex", "receptor_complex"],
    keep="first"
)

topN_per_source_t = (sorted_df.loc[mask]
                   .groupby("source", group_keys=False)
                   .head(5)
                   .sort_values("source"))

In [25]:
df = topN_per_source_endo.copy() 
plot_df_endo = df.loc[:, [
    "source","target","ligand_complex","receptor_complex",
    "delta_score","lr_probs_rt","magnitude_rank_rt"
]].copy()
plot_df_endo["strength_rt"] = 1.0 - plot_df_endo["magnitude_rank_rt"]
d = plot_df_endo["delta_score"]
lim = np.nanpercentile(np.abs(d), 95)
plot_df_endo["delta_clipped"] = d.clip(-lim, lim)
radio.uns["selected_int"] = plot_df_endo

In [26]:
plot_df_endo

In [27]:
plot_df_t = topN_per_source_t.loc[:, [
    "source","target","ligand_complex","receptor_complex",
    "delta_score","lr_probs_rt","magnitude_rank_rt"
]].copy()
plot_df_t["strength_rt"] = 1.0 - plot_df_t["magnitude_rank_rt"]
d = plot_df_t["delta_score"]
lim = np.nanpercentile(np.abs(d), 95)
plot_df_t["delta_clipped"] = d.clip(-lim, lim)
radio.uns["selected_int"] = plot_df_t

In [28]:
plot_df_t

In [29]:
order = ["delta_score", "d_mag", "d_prob"]
sorted_df = top_tumor_caf.sort_values(order, ascending=[False, False, False])
mask = ~sorted_df.duplicated(
    subset=["source", "target", "ligand_complex", "receptor_complex"],
    keep="first"
)

topN_per_source_caf = (sorted_df.loc[mask]
                   .groupby("source", group_keys=False)
                   .head(5)
                   .sort_values("source"))

plot_df_caf = topN_per_source_caf.loc[:, [
    "source","target","ligand_complex","receptor_complex",
    "delta_score","lr_probs_rt","magnitude_rank_rt"
]].copy()
plot_df_caf["strength_rt"] = 1.0 - plot_df_caf["magnitude_rank_rt"]
d = plot_df_caf["delta_score"]
lim = np.nanpercentile(np.abs(d), 95)
plot_df_caf["delta_clipped"] = d.clip(-lim, lim)
radio.uns["selected_int"] = plot_df_caf

In [30]:
plot_df_caf

In [ ]:
from matplotlib.colors import to_hex
from pycirclize import Circos

In [39]:
import yaml

with open("config_color_scheme.yml", "r") as f:
    scheme = yaml.safe_load(f)

In [40]:
celltype_color = scheme['palettes']['Level_4']

# split in two: endo and t cells

In [31]:
adata.obs.groupby('Is_Core')['Dataset'].unique().explode()

In [33]:
!pip install pycirclize

In [34]:
from matplotlib import cm
from matplotlib.colors import to_hex
from pycirclize import Circos
from pycirclize.parser import Matrix
from matplotlib.lines import Line2D

def plot_chord(
    plot_df: pd.DataFrame,
    *,
    value_col="delta_score",
    source_col="source",
    target_col="target",
    ligand_col="ligand_complex",
    receptor_col="receptor_complex",
    celltype_colors=None,       # dict like {"Tumor":"#...", "Endo":"#...", ...}; auto-made if None
    cmap_name="tab20",
    small_gap=1,
    big_gap=3,
    r_outer0=94,
    r_outer1=100,
    label_size=8,
    group_label_size=8,
    link_alpha=0.35,
    link_lw=0.4,
    figsize=(20, 20),
    dpi=300,
    show_gene_labels=False,
    name_sectors=True,
    outpath="radio_circle.png", # set to None to skip saving
):
    """
    Build a chord diagram of ligand(release) -> receptor(receive), grouped by source/target cell types.

    Inputs:
      plot_df: DataFrame with columns [ligand_complex, receptor_complex, delta_score, source, target]
      value_col:     column for link thickness/weight (e.g., "delta_score")
      *_col params:  column names for source/target/ligand/receptor

    Returns:
      fig, circos, edges, nodes
    """

    # Unique node ids per (group, gene)
    edges = plot_df[[ligand_col, receptor_col, value_col, source_col, target_col]].copy()
    edges.columns = ["from_gene", "to_gene", "value", "source_cell", "target_cell"]
    
    edges["from_id"] = edges["from_gene"] + " | " + edges["source_cell"]
    edges["to_id"]   = edges["to_gene"]   + " | " + edges["target_cell"]
    
    # Nodes table = one row per (group, gene)
    lig = edges[["from_id","from_gene","source_cell"]].drop_duplicates() \
            .rename(columns={"from_id":"node","from_gene":"gene","source_cell":"group"})
    rec = edges[["to_id","to_gene","target_cell"]].drop_duplicates() \
            .rename(columns={"to_id":"node","to_gene":"gene","target_cell":"group"})
    nodes = pd.concat([lig, rec], ignore_index=True)
    
    # Order and spacing
    nodes = nodes.sort_values(["group","gene"]).reset_index(drop=True)
    order = nodes["node"].tolist()
    space = [(big_gap if g1!=g2 else small_gap)
             for g1,g2 in zip(nodes["group"], nodes["group"].shift(-1))]
    if space: space[-1] = big_gap
    
    # Colors
    cell_types = nodes["group"].unique().tolist()
    if celltype_colors is None:
        cmap = cm.get_cmap(cmap_name, max(20, len(cell_types)))
        celltype_colors = {ct: to_hex(cmap(i % cmap.N)) for i, ct in enumerate(cell_types)}
    node_color = {row.node: celltype_colors[row.group] for _, row in nodes.iterrows()}
    
    # # Link color by sender group
    sender_of_node = dict(zip(lig["node"], lig["group"]))
    # def link_kws_handler(from_label, to_label):
    #     ct = sender_of_node.get(from_label)
    #     return dict(fc=celltype_colors.get(ct, "#999999"), alpha=link_alpha, zorder=0)
    # choose a per-node padding (constant or based on label length)
    pad_per_node = 2  # try 0.5–2.0
    pad = (
        nodes.assign(value=pad_per_node)
             .rename(columns={"node":"from_id","gene":"from_gene","group":"source_cell"})
    )
    pad["to_id"] = pad["from_id"]
    pad["to_gene"] = pad["from_gene"]
    pad["target_cell"] = pad["source_cell"]
    
    edges = pd.concat([edges, pad[edges.columns]], ignore_index=True)
    
    # hide self-links (alpha=0, no edge)
    def link_kws_handler(from_label, to_label):
        if from_label == to_label:
            return dict(alpha=0, lw=0)  # invisible padding link
        ct = sender_of_node.get(from_label)
        return dict(fc=celltype_colors.get(ct, "#999999"), alpha=link_alpha, zorder=0)
    # Build matrix with the unique node ids
    matrix = Matrix.parse_fromto_table(
        edges[["from_id","to_id","value"]].rename(columns={"from_id":"from","to_id":"to"})
    )
    
    # Create circos using node ids, colored by group
    circos = Circos.chord_diagram(
        matrix,
        order=order,
        space=space,
        r_lim=(r_outer0, r_outer1),
        cmap=node_color,
        label_kws=None,                             # <-- turn off built-in labels
        link_kws=dict(direction=1, ec="black", lw=link_lw),
        link_kws_handler=link_kws_handler,
    )
    if show_gene_labels:
        label_r = (r_outer0 + r_outer1) / 2
        for s in circos.sectors:
            gene_only = s.name.split(" | ")[0]
            s.text(gene_only, r=label_r + 3, size=label_size, orientation='vertical')

    # (B) ONE label per cell type (group)
    if name_sectors:
        name2sector = {s.name: s for s in circos.sectors}
        for ct, gdf in nodes.groupby("group", sort=False):
            mid_node = gdf["node"].iloc[len(gdf) // 2]
            sector = name2sector[mid_node]
            sector.text(ct, r=r_outer1 + 10, size=group_label_size, orientation="horizontal", )
    
    fig = circos.plotfig(figsize=figsize, dpi=dpi)
    # keep only those that exist in your palette
    legend_labels = plot_df.source.unique().tolist() + plot_df.target.unique().tolist() 
    legend_labels = [lab for lab in legend_labels if lab in celltype_colors]
    
    handles = [
        Line2D([0], [0], marker='o', linestyle='None', markersize=10,
               markerfacecolor=celltype_colors[lab], markeredgecolor='none')
        for lab in legend_labels
    ]
    
    # add some bottom margin so the legend fits under the chord
    plt.subplots_adjust(bottom=0.13)
    
    # place legend centered below the axes, spanning multiple columns
    fig.legend(
        handles, legend_labels,
        loc="lower center",
        ncol=6,                # tweak column count to control wrapping
        frameon=False,
        handletextpad=0.6,
        columnspacing=1.2,
        borderaxespad=0.5,
        bbox_to_anchor=(0.5, -0.1)  # y can be slightly negative if you need more room
    )
    ax = fig.axes[0]
    for t in ax.texts:
        if " | " in t.get_text():
            t.set_visible(False)
    plt.show()
    if outpath:
        fig.savefig(outpath, dpi=dpi, bbox_inches="tight")

    return fig, circos, edges, nodes

In [35]:
pwd

In [36]:
plot_df_endo

In [37]:
plot_df_t

In [ ]:
fig, circos, edges, nodes = plot_chord(plot_df_endo, celltype_colors=celltype_color,big_gap=1.33, small_gap=0.20, outpath='Radio_CCC/tumour_endo.png', show_gene_labels=True)

In [41]:
fig, circos, edges, nodes = plot_chord(plot_df_t, celltype_colors=celltype_color,big_gap=1.33, small_gap=0.25, outpath=None, show_gene_labels=True)

In [ ]:
fig, circos, edges, nodes = plot_chord(plot_df_caf, celltype_colors=celltype_color,big_gap=1.33, small_gap=0.25, outpath='Radio_CCC/tumour_caf.png', show_gene_labels=True)

In [ ]:
fig, circos, edges, nodes = plot_chord(plot_df_endo, celltype_colors=celltype_color,big_gap=1, small_gap=0.20, outpath='Radio_CCC/tumour_endo_no_label.png', 
                                       show_gene_labels=False, name_sectors=False, r_outer0=83, r_outer1=97)

In [ ]:
fig, circos, edges, nodes = plot_chord(plot_df_t, celltype_colors=celltype_color,big_gap=1, small_gap=0.20, outpath='Radio_CCC/tumour_t_no_label.png', 
                                       show_gene_labels=False, name_sectors=False, r_outer0=83, r_outer1=97)

In [ ]:
fig, circos, edges, nodes = plot_chord(plot_df_caf, celltype_colors=celltype_color,big_gap=1, small_gap=0.20, outpath='Radio_CCC/tumour_caf_no_label.png', 
                                       show_gene_labels=False, name_sectors=False, r_outer0=83, r_outer1=97)

In [ ]:
fig, circos, edges, nodes = plot_chord(plot_df_endo, celltype_colors=celltype_color,big_gap=2, small_gap=0.33, outpath='Radio_CCC/tumour_endo_no_box.png', 
                                       show_gene_labels=True, name_sectors=False, r_outer0=83, r_outer1=85, label_size=20)

In [ ]:
fig, circos, edges, nodes = plot_chord(plot_df_t, celltype_colors=celltype_color,big_gap=2, small_gap=0.33, outpath='Radio_CCC/tumour_t_no_box.png', 
                                       show_gene_labels=True, name_sectors=False, r_outer0=83, r_outer1=85, label_size=20)

In [ ]:
fig, circos, edges, nodes = plot_chord(plot_df_caf, celltype_colors=celltype_color,big_gap=2, small_gap=0.33, outpath='Radio_CCC/tumour_caf_no_box.png', 
                                       show_gene_labels=True, name_sectors=False, r_outer0=83, r_outer1=85, label_size=20)

In [ ]:
all_cells  = plot_df_endo.source.unique().tolist() + plot_df_endo.target.unique().tolist() + plot_df_t.target.unique().tolist() + plot_df_caf.target.unique().tolist()

In [ ]:
all_cells

In [ ]:
new_dict = {}
for k,v in celltype_color.items():
    if k in all_cells:
        new_dict[k] = v

In [ ]:
celltype_colors = {
    # --- unchanged (malignant) ---
    "Malignant Cell - Pit Like": "#a6cee3",
    "Malignant Cell - Acinar-like": "#b2df8a",
    "Malignant Cell - Epithelial": "#fb9a99",
    "Malignant Cell - Hypoxia": "#fdbf6f",
    "Malignant Cell - EMT": "#cab2d6",
    "Malignant Cell - Highly Proliferative": "#e31a1c",
    "Malignant Cell - Mesenchymal": "#fed683",
    "Malignant Cell - Highly Invasive": "#6a3d9a",
    "Malignant Cell - Senescence": "#1f78b4",
    "Malignant Cell - Apoptotic": "#b15928",

    # --- adjusted non-malignant (deduped) ---
    "Adipocyte": "#ffff99",  # unchanged (unique)
    "CD4+ Memory T Cell": "#9ecae1",
    "CD4+ Naive T Cell": "#c6dbef",
    "CD4+ Th1 Cell": "#6baed6",
    "CD4+ Th17 Cell": "#3182bd",
    "CD4+ Th2 Cell": "#8da0cb",
    "CD4+ Th22 Cell": "#1f7a8c",
    "CD8+ Effector T Cell": "#74add1",
    "CD8+ Terminal Effector T Cell": "#7fb3d5",
    "CD8+ Tissue-Resident Memory T Cell": "#3b8bc2",
    "CD8+ Exhausted T Cell": "#33a02c",  # unchanged (already unique)
    "Double Positive CD4+CD8+ T Cell": "#2b8cbe",
    "γδ T Cell (Vδ1)": "#4c78a8",
    "iCAF": "#a1d99b",
    "Lymphatic Endothelial Cell": "#17BECF",  # unchanged (unique)
    "myCAF": "#74c476",
    "Schwann Cell": "#843c39",               # unchanged (unique)
    "T-reg": "#5b9bd5",
    "Tumor-Associated Endothelial Cell": "#9467BD",  # unchanged (unique)
    "Vascular Endothelial Cell": "#E6A700",          # unchanged (unique)
}

In [ ]:
fig, circos, edges, nodes = plot_chord(plot_df_endo, celltype_colors=celltype_color,big_gap=1.33, small_gap=0.50, 
                                       outpath='Radio_CCC/tumour_endo_legend_bottom.png', 
                                       show_gene_labels=True, name_sectors=False, r_outer0=83, r_outer1=85.5, label_size=12, figsize=(10,10))

In [42]:
fig, circos, edges, nodes = plot_chord(plot_df_t, celltype_colors=celltype_color,big_gap=1.33, small_gap=0.50, 
                                       outpath=None, 
                                       show_gene_labels=True, name_sectors=False, r_outer0=83, r_outer1=85.5, label_size=12, figsize=(10,10))

In [ ]:
fig, circos, edges, nodes = plot_chord(plot_df_caf, celltype_colors=celltype_color,big_gap=1.33, small_gap=0.50, 
                                       outpath='Radio_CCC/tumour_caf_legend_bottom.png', 
                                       show_gene_labels=True, name_sectors=False, r_outer0=83, r_outer1=85.5, label_size=12, figsize=(10,10))

In [44]:
plot_df_t.to_csv('Radio_CCC/plotted_t_mal_interactions.csv')

In [50]:
focus.head()

In [64]:
assert {"d_mag_z", "d_prob_z", "d_logfc_z", "p_rt"}.issubset(focus.columns), (
    "Run the CCC.ipynb cells that build `focus` (through the delta_score/meaningful "
    "cell) in this kernel before running this notebook."
)


In [80]:
weight_schemes = {
    "original (1.0/0.7/0.5)":       (1.0, 0.7, 0.5),
    "equal (1/1/1)":                (1.0, 1.0, 1.0),
    "magnitude-only (1/0/0)":       (1.0, 0.0, 0.0),
    "probability-heavy (0.5/1/0.5)":(0.5, 1.0, 0.5),
    "logfc-heavy (0.5/0.5/1)":      (0.5, 0.5, 1.0),
    "reversed (0.5/0.7/1.0)":       (0.5, 0.7, 1.0),
}

axis_defs = {
    "Tumor/Endo": axes["Tumor/Endo"],
    "Tumor/CAF":  axes["Tumor/CAF"],
    "Tumor/T":    axes["Tumor/T"],
}


def compute_delta_score(df, w_mag, w_prob, w_logfc):
    return w_mag * df["d_mag_z"] + w_prob * df["d_prob_z"] + w_logfc * df["d_logfc_z"]


def select_top5_per_source(df, score_col="delta_score"):
    """Mirrors the exact selection used to build plot_df_endo/_t/_caf in
    CCC.ipynb: sort by score (ties broken by d_mag, d_prob), dedup, top 5
    interactions per source cell type.
    """
    d = (df.sort_values([score_col, "d_mag", "d_prob"], ascending=[False, False, False])
           .drop_duplicates(subset=["source", "target", "ligand_complex", "receptor_complex"], keep="first"))
    return d.groupby("source", group_keys=False).head(5)


def interaction_keys(df):
    return set(zip(df.source, df.target, df.ligand_complex, df.receptor_complex))


## Recompute the top-5-per-source selection under each weight scheme, per axis

In [81]:
baseline_selections = {
    axis_name: select_top5_per_source(meaningful[mask_fn(meaningful)], score_col="delta_score")
    for axis_name, mask_fn in axis_defs.items()
}

rows = []
for scheme_name, (w_mag, w_prob, w_logfc) in weight_schemes.items():
    focus_alt = focus.assign(delta_score_alt=compute_delta_score(focus, w_mag, w_prob, w_logfc))
    # Same gate as the original pipeline (p_rt < 0.05 and delta_score > 0),
    # just applied to the re-weighted score \u2014 this also captures cases where
    # re-weighting changes *which* interactions pass the significance gate,
    # not just their rank.
    meaningful_alt = focus_alt[(focus_alt["p_rt"] < 0.05) & (focus_alt["delta_score_alt"] > 0)]

    for axis_name, mask_fn in axis_defs.items():
        sel_alt = select_top5_per_source(meaningful_alt[mask_fn(meaningful_alt)], score_col="delta_score_alt")

        base_keys = interaction_keys(baseline_selections[axis_name])
        alt_keys = interaction_keys(sel_alt)
        union = base_keys | alt_keys
        jaccard = len(base_keys & alt_keys) / len(union) if union else np.nan

        # Spearman correlation of the (re-weighted) score against the original
        # delta_score, across every interaction in `meaningful` for this axis
        # (not just the top 5) \u2014 a broader check than just the plotted subset.
        base_axis = meaningful[mask_fn(meaningful)][
            ["source", "target", "ligand_complex", "receptor_complex", "delta_score"]]
        alt_axis = meaningful_alt[mask_fn(meaningful_alt)][
            ["source", "target", "ligand_complex", "receptor_complex", "delta_score_alt"]]
        common = base_axis.merge(alt_axis, on=["source", "target", "ligand_complex", "receptor_complex"], how="inner")
        rho = common[["delta_score", "delta_score_alt"]].corr(method="spearman").iloc[0, 1] if len(common) > 2 else np.nan

        rows.append({
            "weight_scheme": scheme_name,
            "axis": axis_name,
            "n_baseline_selected": len(base_keys),
            "n_alt_selected": len(alt_keys),
            "n_overlap": len(base_keys & alt_keys),
            "jaccard_overlap": jaccard,
            "spearman_rho_full_axis": rho,
        })

sensitivity_df = pd.DataFrame(rows)
sensitivity_df


## Summary and figure

In [82]:
summary = (sensitivity_df[sensitivity_df.weight_scheme != "original (1.0/0.7/0.5)"]
           .groupby("weight_scheme")[["jaccard_overlap", "spearman_rho_full_axis"]]
           .mean()
           .sort_values("jaccard_overlap", ascending=False))
print("Average agreement with the manuscript's weights (1.0/0.7/0.5), "
      "across the Tumor/Endo, Tumor/CAF, and Tumor/T axes:")
summary


In [84]:
pivot_jaccard = sensitivity_df.pivot(index="weight_scheme", columns="axis", values="jaccard_overlap")
pivot_rho = sensitivity_df.pivot(index="weight_scheme", columns="axis", values="spearman_rho_full_axis")

fig, axs = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(pivot_jaccard, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=axs[0])
axs[0].set_title("Top-5-per-source overlap (Jaccard)\nvs. manuscript weights (1.0/0.7/0.5)")
sns.heatmap(pivot_rho, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=axs[1])
axs[1].set_title("Spearman rank correlation of delta_score\nvs. manuscript weights (1.0/0.7/0.5)")
plt.tight_layout()
plt.savefig("Radio_CCC/delta_score_weight_sensitivity.png", dpi=300, bbox_inches="tight")
plt.show()

sensitivity_df.to_csv("Radio_CCC/delta_score_weight_sensitivity.csv", index=False)


## Per-malignant-subtype breakdown

The pooled, per-axis Jaccard above averages across all ~10 malignant/tumor
subtypes at once. Since the actual selection is "top 5 interactions **per
malignant subtype**" (`topN_per_source_t`/`_endo`/`_caf` in `CCC.ipynb`), a
pooled axis-level number can hide the real picture \u2014 e.g. one subtype could
be perfectly stable across weight schemes while another is completely
unstable, and the pooled average would just look mediocre for both. This
section scores overlap **per malignant subtype**, matching the actual
plotting unit.


In [69]:
def per_source_jaccard(baseline_df, alt_df):
    """Jaccard overlap of the top-5 selection, computed separately for each
    malignant/tumor subtype (source), rather than pooled across all of them.
    """
    out = {}
    for src in sorted(set(baseline_df.source) | set(alt_df.source)):
        base_keys = interaction_keys(baseline_df[baseline_df.source == src])
        alt_keys = interaction_keys(alt_df[alt_df.source == src])
        union = base_keys | alt_keys
        out[src] = len(base_keys & alt_keys) / len(union) if union else np.nan
    return out


per_source_rows = []
for scheme_name, (w_mag, w_prob, w_logfc) in weight_schemes.items():
    if scheme_name == "original (1.0/0.7/0.5)":
        continue
    focus_alt = focus.assign(delta_score_alt=compute_delta_score(focus, w_mag, w_prob, w_logfc))
    meaningful_alt = focus_alt[(focus_alt["p_rt"] < 0.05) & (focus_alt["delta_score_alt"] > 0)]

    for axis_name, mask_fn in axis_defs.items():
        sel_alt = select_top5_per_source(meaningful_alt[mask_fn(meaningful_alt)], score_col="delta_score_alt")
        jaccard_by_source = per_source_jaccard(baseline_selections[axis_name], sel_alt)
        for source, jaccard in jaccard_by_source.items():
            per_source_rows.append({
                "weight_scheme": scheme_name,
                "axis": axis_name,
                "source": source,
                "jaccard_overlap": jaccard,
            })

per_source_df = pd.DataFrame(per_source_rows)
per_source_df.to_csv("Radio_CCC/delta_score_weight_sensitivity_per_source.csv", index=False)
per_source_df


In [70]:
# Which malignant subtypes are most/least stable, averaged across the 5
# non-original weight schemes and all three axes -- this is the actionable
# list: subtypes near the bottom are the ones whose plotted interactions are
# most weight-dependent and worth double-checking (or footnoting) in the figure.
stability_by_source = (per_source_df.groupby("source")["jaccard_overlap"]
                        .agg(["mean", "min", "count"])
                        .sort_values("mean"))
stability_by_source


In [72]:
for axis_name in axis_defs:
    pivot = per_source_df[per_source_df.axis == axis_name].pivot(
        index="weight_scheme", columns="source", values="jaccard_overlap")
    fig, ax = plt.subplots(figsize=(12, 6))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="viridis", vmin=0, vmax=1, ax=ax)
    ax.set_title(f"{axis_name}: top-5-per-malignant-subtype overlap vs. manuscript weights")
    plt.tight_layout()
    plt.savefig(f"Radio_CCC/delta_score_weight_sensitivity_{axis_name.replace('/', '_')}_per_source.png",
                dpi=300, bbox_inches="tight")
    plt.show()


In [73]:
meaningful[axes["Tumor/T"](meaningful)].groupby("source").size().sort_values()

In [74]:
def top_gap(df, mask_fn, n=5):
    out = {}
    d = df[mask_fn(df)]
    for src, grp in d.groupby("source"):
        scores = grp["delta_score"].sort_values(ascending=False).values
        out[src] = scores[n-1] - scores[n] if len(scores) > n else np.nan
    return pd.Series(out).sort_values()

top_gap(meaningful, axes["Tumor/T"])